In [ ]:
import serial
import MotoricaInterface as mi ; assert mi.__version__ == '0.0.8' # should be updated each time changes occur
from MotoricaInterface.ubi4 import ubi4_enum_description, ubi4_p2p_high_level_port
from MotoricaInterface.ubi4 import ubi4_p2p_high_level_port_pack
import time
import numpy as np
import matplotlib.pyplot as plt

%matplotlib ipympl

%load_ext autoreload
%autoreload 2


In [2]:
mi.utils.list_devices()

,device,name,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,/dev/cu.BLTH,cu.BLTH,n/a,n/a,NaN,NaN,None,None,None,None,None
1,/dev/cu.BoseQC35II,cu.BoseQC35II,n/a,n/a,NaN,NaN,None,None,None,None,None
2,/dev/cu.Bluetooth-Incoming-Port,cu.Bluetooth-Incoming-Port,n/a,n/a,NaN,NaN,None,None,None,None,None
3,/dev/cu.usbmodem1234561,cu.usbmodem1234561,Espressif Device,USB VID:PID=303A:4001 SER=123456 LOCATION=20-3,12346.0,16385.0,123456,20-3,Espressif Systems,Espressif Device,None


In [3]:
   
ubi4_stack = mi.ubi4.Ubi4Stack('/dev/cu.usbmodem1234561')
ubi4_stack.initialize_stack()

display(ubi4_stack)

for sub_device in ubi4_stack.sub_devices_array:
    print('--------------SUBDEVICE------------------')
    display(sub_device)

assert "DCE_OMG_MODULE" in [ubi4_enum_description.device_codes_enum(sub_device.DeviceCode).name 
                            for sub_device in ubi4_stack.sub_devices_array], "DCE_OMG_MODULE not found"

--------------------------------
Stopping streaming
Sending: ['0x20', '0x08', '0x00', '0x01', '0x00', '0xbc', '0x00', '0x02']
Should be: [0x20, 0x08, 0x00, 0x01, 0x00, 0x00, 0x00, 0x02]
Streaming stopped
********************************
Serial flushed
Serial flushed
--------------------------------
Requesting first info from the stack
Sending: ['0x00', '0x01', '0x00', '0x02', '0x00', '0x78', '0x00', '0x01', '0x02']
CRC check failed: pack_crc=20, calc_crc=78
FullInitializeConnectionStruct loaded
Full initialize structure:
DeviceName: UBIv4_OPTOBOX
DeviceVersion: 1
DeviceSubVersion: 0
DeviceLabel: t
DeviceType: 1
DeviceCode: 1
DeviceAddress: 0
DeviceUUID_Prefix: CPU MODULE
DeviceUUID: 17
ParametrsNum: 4
SubDeviceNum: 3
ProgramType: 2
DefaultPort: 0
********************************
--------------------------------
Requesting device parameters
Requesting device parameters: ['0x00', '0x01', '0x00', '0x03', '0x00', '0x31', '0x00', '0x02', '0x00', '0x0a']
CRC check failed: pack_crc=127, calc_

UBI4Stack
ser=/dev/cu.usbmodem1234561
DeviceName: UBIv4_OPTOBOX
DeviceVersion: 1
DeviceSubVersion: 0
DeviceLabel: t
DeviceType: 1
DeviceCode: 1
DeviceAddress: 0
DeviceUUID_Prefix: CPU MODULE
DeviceUUID: 17
ParametrsNum: 4
SubDeviceNum: 3
ProgramType: 2
DefaultPort: 0
3 subdevices found
Parameters:
	(PDCE_SIMPLE_COMMAND:1 bytes)
	(PDCE_ACTION_REQUEST:6 bytes)
	(PDCE_SELECT_PROFILE:1 bytes)
	(PDCE_INTERFECE_ERROR_COUNTER:1 bytes)

--------------SUBDEVICE------------------


DeviceType=37 : DTE_OMG_MULTY_CH
DeviceCode=7 : DCE_OMG_MODULE
DeviceRole=7 : DRE_HAND_OPEN_SENSOR
DeviceVersion=1
DeviceSubVersion=0
DeviceAddress=10
ParametersNum=9
SubDeviceNum=0
DefaultPort=2
Parameters:
	(PDCE_SIMPLE_COMMAND:4 bytes)
	(PDCE_ACTION_REQUEST:6 bytes)
	(PDCE_EMG_CH_1_3_VAL:2 bytes)
	(PDCE_EMG_ENV_E_VAL:8 bytes)
	(PDCE_DMS_OUTPUT:1 bytes)
	(PDCE_OPTIC_LEARNING_DATA:176 bytes)
	(PDCE_OPTIC_CHECKPOINT_DATA_RECEIVER:204 bytes)
	(PDCE_GENERIC_2:6 bytes)
	(PDCE_GENERIC_1:1 bytes)

--------------SUBDEVICE------------------


DeviceType=81 : DTE_BMS
DeviceCode=5 : DCE_BMS
DeviceRole=6 : DRE_BMS
DeviceVersion=1
DeviceSubVersion=0
DeviceAddress=7
ParametersNum=2
SubDeviceNum=0
DefaultPort=1
Parameters:
	(PDCE_BMS_STATUS_COMBINED_PARAM:4 bytes)
	(PDCE_DATE_AND_TIME:8 bytes)

--------------SUBDEVICE------------------


DeviceType=68 : DTE_DISPLAY_BUTTONS
DeviceCode=6 : DCE_GUI
DeviceRole=2 : DRE_DISPLAY
DeviceVersion=1
DeviceSubVersion=0
DeviceAddress=9
ParametersNum=17
SubDeviceNum=0
DefaultPort=1
Parameters:
	(PDCE_ACTION_REQUEST:6 bytes)
	(PDCE_GESTURE_GROUP:16 bytes)
	(PDCE_SELECT_GESTURE:1 bytes)
	(PDCE_SELECT_PROFILE:1 bytes)
	(PDCE_GLOBAL_FORCE:1 bytes)
	(PDCE_GLOBAL_SENSITIVITY:1 bytes)
	(PDCE_GLOBAL_THRESHOLD:1 bytes)
	(PDCE_OPEN_CLOSE_THRESHOLD:4 bytes)
	(PDCE_BMS_STATUS_COMBINED_PARAM:4 bytes)
	(PDCE_EMG_CH_1_3_VAL:2 bytes)
	(PDCE_EMG_CH_1_3_GAIN:2 bytes)
	(PDCE_EMG_ENV_E_VAL:8 bytes)
	(PDCE_DATE_AND_TIME:8 bytes)
	(PDCE_ENERGY_SAVE_MODE:1 bytes)
	(PDCE_LEFT_RIGHT_HAND_MODE:4 bytes)
	(PDCE_SIMPLE_COMMAND:0 bytes)
	(PDCE_SIMPLE_COMMAND:0 bytes)

In [4]:
all_params_names = [ubi4_enum_description.parameter_data_code_enum(p.DataCode).name 
                    if p.DataCode in ubi4_enum_description.parameter_data_code_enum._value2member_map_ 
                    else str(p.DataCode) for p in ubi4_stack.sub_devices_array[0].parsed_parameters]
local_index = all_params_names.index('PDCE_OPTIC_CHECKPOINT_DATA_RECEIVER')
local_param = ubi4_stack.sub_devices_array[0].parsed_parameters[local_index]
chunk_size = local_param.parameter_size - 4  # size of each chunk in bytes

local_index, chunk_size

(6, 200)

In [5]:
omg_device = ubi4_stack.sub_devices_array[0]
[{p.ID: ubi4_enum_description.parameter_data_code_enum(p.DataCode).name} 
 if p.DataCode in ubi4_enum_description.parameter_data_code_enum._value2member_map_ 
 else {p.ID: str(p.DataCode)} 
 for p in omg_device.parsed_parameters]


[{0: 'PDCE_SIMPLE_COMMAND'},
 {1: 'PDCE_ACTION_REQUEST'},
 {2: 'PDCE_EMG_CH_1_3_VAL'},
 {3: 'PDCE_EMG_ENV_E_VAL'},
 {4: 'PDCE_DMS_OUTPUT'},
 {5: 'PDCE_OPTIC_LEARNING_DATA'},
 {6: 'PDCE_OPTIC_CHECKPOINT_DATA_RECEIVER'},
 {7: 'PDCE_GENERIC_2'},
 {8: 'PDCE_GENERIC_1'}]

# Request status for data transfer

In [ ]:
ubi4_stack.start_parsing_thread()
ubi4_stack.start_streaming()

--------------------------------
Starting parsing thread
Serial flushed
Parsing thread started successfully
********************************
--------------------------------
Starting streaming
Sending: ['0x20', '0x08', '0x00', '0x01', '0x00', '0x5e', '0x00', '0x01']
Should be: [0x20, 0x08, 0x00, 0x01, 0x00, 0x00, 0x00, 0x01]
Streaming started
********************************


b''

Exception in thread Thread-4 (parsing_thread):
Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/opto/lib/python3.11/site-packages/serial/serialposix.py", line 575, in read
    buf = os.read(self.fd, size - len(read))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 6] Device not configured

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/opto/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/anaconda3/envs/opto/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/anaconda3/envs/opto/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/mike/src/pyMotoricaInterface/MotoricaInterface/ubi4/ubi4_stack.py", line 300, in parsing_thread
    in_byte = self.ser.read(1)
              ^^^^^^^^^^^^^^^^
  F

# Send data to device

In [7]:
megabyte = 1024 * 1024  # 1 megabyte in bytes
num_chunks = megabyte // chunk_size  # number of chunks in 1 megabyte
max_uint16 = (1 << 16) - 1

num_chunks, max_uint16

(5242, 65535)

In [8]:
import struct
import pandas as pd

def construct_pack_for_transfer(pack_chunk: bytearray,
                               command: int = 0,
                               pack_index_in_series: int = 0):
    
    n_bytes = len(pack_chunk)
    assert n_bytes < 50, "pack_chunk size must be less than 50 bytes"

    header_bytes = struct.pack('<BBH', command, n_bytes, pack_index_in_series) 
    pack_payload = header_bytes + pack_chunk + bytearray([0x00]*(64 - n_bytes - len(header_bytes)))

    pack_header = ubi4_p2p_high_level_port.construct_p2p_high_level_port_header(port = 0,
                                                                                bridge = 0, 
                                                                                waiting_answer = 1, 
                                                                                request_type = 1,
                                                                                package_code=local_index | 0x80, 
                                                                                package_id=0, 
                                                                                package_size=len(pack_payload), 
                                                                                crc=mi.get_crc(pack_payload, len(pack_payload)), 
                                                                                address=0x0A)

    pack= bytearray(pack_header) + bytearray(pack_payload)

    return pack    


packs = []

packs.append(construct_pack_for_transfer('temp3.ckpt'.encode('utf-8'), 
                                        command=1, 
                                        pack_index_in_series=1))

packs.append(construct_pack_for_transfer('first part'.encode('utf-8'), 
                                        command=2, 
                                        pack_index_in_series=2))
packs.append(construct_pack_for_transfer(' second part'.encode('utf-8'), 
                                        command=2, 
                                        pack_index_in_series=3))
packs.append(construct_pack_for_transfer(' third part'.encode('utf-8'), 
                                        command=2, 
                                        pack_index_in_series=4))
packs.append(construct_pack_for_transfer(' finish'.encode('utf-8'), 
                                        command=2,
                                        pack_index_in_series=5))

packs.append(construct_pack_for_transfer(''.encode('utf-8'), 
                                        command=3,
                                        pack_index_in_series=6))

packs


[bytearray(b'`\x86\x00@\x00\xc5\n\x01\n\x01\x00temp3.ckpt\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'),
 bytearray(b'`\x86\x00@\x000\n\x02\n\x02\x00first part\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'),
 bytearray(b'`\x86\x00@\x00\x90\n\x02\x0c\x03\x00 second part\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'),
 bytearray(b'`\x86\x00@\x00\x1a\n\x02\x0b\x04\x00 third part\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00

In [9]:
CKPT_PACK_TIMEOUT_MS = 1000

ubi4_state_history = []

for ipack, pack in enumerate(packs[:]):
    print('Sending Header:', bin(pack[0]), 
          'Payload:', list(map(lambda x: f'0x{x:02x}', pack)))

    # TODO: stream parsing
    ubi4_stack.__ubi4_state_history__.clear()
    ubi4_stack.pack_history.clear()
    ubi4_stack.ser.write(pack)
    
    ts_send = time.time()
    
    good_flag = 0
    while (time.time() - ts_send) < CKPT_PACK_TIMEOUT_MS/1000:
        if len(ubi4_stack.__ubi4_state_history__) > 0:
            ts_state, ubi4_state = ubi4_stack.__ubi4_state_history__.popleft()
            ubi4_state_history.append((ts_state, ubi4_state))
            if ('DCE_OMG_MODULE', 'PDCE_GENERIC_2') in ubi4_state:
                values = ubi4_state[('DCE_OMG_MODULE', 'PDCE_GENERIC_2')]
                print(values)
                
                if (values["OPTIC_RECEIVER_HAS_NEW_FLAG"] == 0) and \
                    (values["pack_index_in_series"] == ipack+1) and \
                    (values["sd-res"] == 'FR_OK'):
                    print('next pack')
                    good_flag = 1
                    break

        else:
            time.sleep(0.01)
    if good_flag == 0:
        raise Exception('Timeout')

    # time.sleep(1.0)


Sending Header: 0b1100000 Payload: ['0x60', '0x86', '0x00', '0x40', '0x00', '0xc5', '0x0a', '0x01', '0x0a', '0x01', '0x00', '0x74', '0x65', '0x6d', '0x70', '0x33', '0x2e', '0x63', '0x6b', '0x70', '0x74', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00', '0x00']
{'OPTIC_RECEIVER_HAS_NEW_FLAG': 1, 'inOpticsPack-n_bytes': 10, 'pack_index_in_series': 1, 'sd-res': 'FR_OK', 'sd-bytes_written': 255}
{'OPTIC_RECEIVER_HAS_NEW_FLAG': 0, 'inOpticsPack-n_bytes': 10, 'pack_index_in_series': 1, 'sd-res': 'FR_OK', 'sd-bytes_written': 255}
next pack
Sending Header: 0b1100000 Payload: ['0x60', '0x86', '0x00', '0x40', '0x00', '0x30', '0x0a', '0x02', '0x0a', '0x02', '0

In [11]:
df_states = pd.DataFrame({ts: state for ts, state in ubi4_state_history}).T
df_states.index -= df_states.index[0]
df_PDCE_GENERIC_2 = pd.DataFrame(df_states[('DCE_OMG_MODULE', 'PDCE_GENERIC_2')].to_list())
df_PDCE_GENERIC_2

,OPTIC_RECEIVER_HAS_NEW_FLAG,inOpticsPack-n_bytes,pack_index_in_series,sd-res,sd-bytes_written
0,1,10,1,FR_OK,255
1,0,10,1,FR_OK,255
2,1,10,2,255,255
3,1,10,2,FR_OK,10
4,0,10,2,FR_OK,10
5,1,12,3,FR_OK,12
6,0,12,3,FR_OK,12
7,1,11,4,FR_OK,11
8,0,11,4,FR_OK,11
9,1,7,5,FR_OK,7
